## Webpage Extraction and Embedding (PolyU CUS)

### 1. Extracting raw text data

In [12]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader, SitemapLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [13]:
cus_URL = "https://www.polyu.edu.hk/cus/"
start_idx, stop_idx = 6710, -850

docs = []

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "html.parser")           # Strip the html syntax (using html.parser for better compatibility)
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess whitespaces
    return text

cus_loader = RecursiveUrlLoader(
    max_depth=6,
    url=cus_URL,
    base_url=cus_URL,
    prevent_outside=True,
    exclude_dirs=[
        cus_URL+"about-ous",
        cus_URL+"about-cus",
        cus_URL+"Sitemap", 
        cus_URL+"sitemap",
        cus_URL+"Search-Result", 
        cus_URL+"search-result", 
        cus_URL+"internal",
        cus_URL+"-",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = cus_loader.lazy_load()
for doc in docs_lazy:
    print(doc.metadata.get('source'))
    
    doc.page_content = doc.page_content[start_idx:stop_idx]
    docs.append(doc)

https://www.polyu.edu.hk/cus/
https://www.polyu.edu.hk/cus/non-local-gur-study/non-local-study-fund/non-local-study-fund/
https://www.polyu.edu.hk/cus/non-local-gur-study/non-local-car-subjects/non-local-car-subjects/
https://www.polyu.edu.hk/cus/undergraduate-studies-support/student/thank-you-note/
https://www.polyu.edu.hk/cus/undergraduate-studies-support/student/student-learning-checkpoints/
https://www.polyu.edu.hk/cus/Undergraduate-Studies-Support/Student/Student-Learning-Checkpoints/Enrichment-for-my-study-life-at-PolyU?sc_lang=en
https://www.polyu.edu.hk/cus/Undergraduate-Studies-Support/Student/Student-Learning-Checkpoints/Challenges-for-my-study-life-at-PolyU?sc_lang=en
https://www.polyu.edu.hk/cus/student/senior-year-intakes-and-articulation-degree-programme/academic-integrity/
https://www.polyu.edu.hk/cus/Undergraduate-Studies-Support/Student/Student-Learning-Checkpoints/my-PolyU-life-Starts?sc_lang=en
https://www.polyu.edu.hk/cus/staff/academic-integrity/
https://www.polyu.

In [14]:
print(f"Extracted number of webpages in CUS: {len(docs)}")
print(docs[33].page_content[:])
print(docs[33].metadata.get('source'))

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in CUS: 46
                            


                                Personal Information Collection Statement
                            

Quick Access

Start main content


													Home
												


													Student
												


													4-Year Undergraduate Student
												


													Credit Transfer
												

Credit Transfer

Students should submit an application for credit transfer upon your initial enrolment on the programme or before the end of the add/drop period of the first semester of your first year of study.

All credits transferred will be counted for satisfying the award requirements. Transferred credits are normally not counted for meeting the requirements of more than one degree.

Some programmes may accept applicants holding advanced qualification. If you have an advanced qualification relevant to the programme enrolled, you may be allowed to take fewer credits than the programme normally requires. Howeve

"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1)
    chunk.metadata["chunk_id"] = f"PolyU_CUS_{url_end}_chunk_{i}"

    chunk.page_content = f"--- PolyU CUS Website URL: {source} ---\n\n{chunk.page_content}"

In [16]:
print(chunks[20])

page_content='--- PolyU CUS Website URL: https://www.polyu.edu.hk/cus/Undergraduate-Studies-Support/Student/Student-Learning-Checkpoints/Enrichment-for-my-study-life-at-PolyU?sc_lang=en ---

Contact Us
                            


                                Personal Information Collection Statement
                            

Quick Access

Start main content


													Home
												


													Undergraduate Studies Support
												


													Student
												


													Student Learning Checkpoints
												


													Enrichment for my study life at PolyU
												

Enrichment for my study life at PolyU

University life means more than studying. There will be various opportunities for enriching your university life. You should consider your interest, ability and capacity, career inspiration and study plan when taking up the enrichment opportunities. Also, you should consult your Academic Advisor before making the decision.

☑ Secondary Ma

### 3. Document Embedding in Chroma

In [21]:
SINGLE = False # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

In [22]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

In [23]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i + 0)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Added 141 chunks into ChromaDB to academic_documents


### 4. Simple Testing

In [20]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "What is the general university requirement for undergraduate student?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: --- PolyU CUS Website URL: https://www.polyu.edu.hk/cus/student/4-year-undergraduate-student/general-university-requirements/ ---

Contact Us
                            


                                Personal Information Collection Statement
                            

Quick Access

Start main content


													Home
												


													Student
												


													4-Year Undergraduate Student
												


													General University Requirements (GUR)
												

General University Requirements (GUR)

GUR for 4-Year Undergraduate Student 

 

Admitted in 2021/22 or before

Freshman Seminar

Language & Communication Requirements

Leadership & Intra-Personal Development

Cluster-Area Requirements

Service-Learning

Healthy Lifestyle

 

Admitted from 2022/23

Artificial Intelligence and Data Analytics Requirement

Innovation and Entrepreneurship Requirement

Language & Communication Requirements

Leadership Education and Development

Cluster-Ar

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_82105/1653767187.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(
